In [1]:
from google.colab import drive
drive.mount ('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pytesseract
import os
import pandas as pd
import pytesseract
from PIL import Image

In [10]:
!apt-get update -qq
!apt-get install -y tesseract-ocr-hin tesseract-ocr-asm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tesseract-ocr-asm tesseract-ocr-hin
0 upgraded, 2 newly installed, 0 to remove and 30 not upgraded.
Need to get 2,334 kB of archives.
After this operation, 3,198 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-asm all 1:4.00~git30-7274cfa-1.1 [1,421 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-hin all 1:4.00~git30-7274cfa-1.1 [913 kB]
Fetched 2,334 kB in 1s (1,626 kB/s)
Selecting previously unselected package tesseract-ocr-asm.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-asm_1%3a4.00~git30-727

In [24]:
# ============================================================
# OCR ONLY IMAGE ROWS - PRESERVE ALL MANUALLY ENTERED TEXT
# ============================================================

import os
import pandas as pd
from PIL import Image
import pytesseract
from openpyxl import load_workbook
from openpyxl.styles import Alignment

# ============================================================
# 1. PATHS
# ============================================================

EXCEL_FILE = "/content/drive/MyDrive/Flood_Dataset/dataset.xlsx"

IMAGE_FOLDER = "/content/drive/MyDrive/Flood_Dataset/images"

# New output file - your original file will NOT be changed
OUTPUT_FILE = "/content/drive/MyDrive/Flood_Dataset/dataset_OCR.xlsx"

# Optional CSV output
CSV_FILE = "/content/drive/MyDrive/Flood_Dataset/dataset_OCR.csv"


# ============================================================
# 2. READ EXISTING EXCEL FILE
# ============================================================

print("Reading original Excel workbook...")

df = pd.read_excel(EXCEL_FILE)

print("Original rows:", len(df))
print("Columns:", list(df.columns))


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Id",
    "Source",
    "Text",
    "Image",
    "Extracted Text",
    "Language",
    "Code-Mixed",
    "Translated Text"
]

for col in required_columns:
    if col not in df.columns:
        df[col] = ""

# Keep the desired column order
df = df[required_columns]


# ============================================================
# 4. CLEAN NaN VALUES
# ============================================================

df["Text"] = df["Text"].fillna("")
df["Image"] = df["Image"].fillna("")
df["Extracted Text"] = df["Extracted Text"].fillna("")


# ============================================================
# 5. PROCESS ONLY ROWS HAVING AN IMAGE
# ============================================================

print("\nStarting OCR...")
print("--------------------------------------------")

for index, row in df.iterrows():

    image_name = str(row["Image"]).strip()

    # --------------------------------------------------------
    # TEXT HEADLINE ROW
    # --------------------------------------------------------
    # If Image column is empty:
    #   Text remains exactly as manually entered
    #   Image remains empty
    #   Extracted Text remains empty
    # --------------------------------------------------------

    if image_name == "" or image_name.lower() == "nan":

        print(
            f"Row {index + 2}: TEXT ROW → OCR skipped"
        )

        # DO NOT TOUCH the manually entered Text
        df.at[index, "Image"] = ""
        df.at[index, "Extracted Text"] = ""

        continue


    # --------------------------------------------------------
    # IMAGE ROW
    # --------------------------------------------------------

    image_path = os.path.join(IMAGE_FOLDER, image_name)

    print(
        f"Row {index + 2}: IMAGE ROW → {image_name}"
    )

    # Check whether image exists
    if not os.path.exists(image_path):

        print("   WARNING: Image not found!")
        print("   Expected:", image_path)

        # Keep Image name but leave OCR empty
        df.at[index, "Extracted Text"] = ""

        continue


    # --------------------------------------------------------
    # RUN OCR
    # --------------------------------------------------------

    try:

        image = Image.open(image_path)

        extracted_text = pytesseract.image_to_string(
            image,
            lang="eng+hin+asm",
            config="--psm 6"
        ).strip()

        df.at[index, "Extracted Text"] = extracted_text

        # For image rows Text must remain empty
        df.at[index, "Text"] = ""

        print("   OCR completed")

        if extracted_text:
            print("   Result:", extracted_text[:100])
        else:
            print("   No text detected")

    except Exception as e:

        print("   OCR ERROR:", e)
        df.at[index, "Extracted Text"] = ""


# ============================================================
# 6. SAVE DATA
# ============================================================

print("\nSaving OCR Excel file...")

df.to_excel(
    OUTPUT_FILE,
    index=False
)

print("Excel saved:")
print(OUTPUT_FILE)


# ============================================================
# 7. APPLY WRAP TEXT AND COLUMN WIDTHS
# ============================================================

print("\nApplying Excel formatting...")

wb = load_workbook(OUTPUT_FILE)
ws = wb.active

# ------------------------------------------------------------
# Column widths
# ------------------------------------------------------------

column_widths = {
    "A": 8,     # Id
    "B": 18,    # Source
    "C": 60,    # Text
    "D": 25,    # Image
    "E": 60,    # Extracted Text
    "F": 15,    # Language
    "G": 15,    # Code-Mixed
    "H": 60     # Translated Text
}

for column, width in column_widths.items():
    ws.column_dimensions[column].width = width


# ------------------------------------------------------------
# WRAP TEXT
# ------------------------------------------------------------

for row in ws.iter_rows():

    for cell in row:

        cell.alignment = Alignment(
            wrap_text=True,
            vertical="top"
        )


# ------------------------------------------------------------
# Save formatted workbook
# ------------------------------------------------------------

wb.save(OUTPUT_FILE)

print("Formatting applied successfully!")


# ============================================================
# 8. ALSO CREATE CSV
# ============================================================

df.to_csv(
    CSV_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nCSV also saved:")
print(CSV_FILE)


# ============================================================
# 9. SUMMARY
# ============================================================

text_rows = 0
image_rows = 0

for _, row in df.iterrows():

    if str(row["Image"]).strip() == "":
        text_rows += 1
    else:
        image_rows += 1

print("\n============================================")
print("PROCESS COMPLETED")
print("============================================")
print("Total rows:", len(df))
print("Text headline rows:", text_rows)
print("Image rows:", image_rows)
print("Excel output:", OUTPUT_FILE)
print("CSV output:", CSV_FILE)

display(df)

Reading original Excel workbook...
Original rows: 33
Columns: ['Id', 'Source', 'Text', 'Image', 'Extracted Text', 'Language', 'Code-Mixed', 'Translated Text']

Starting OCR...
--------------------------------------------
Row 2: IMAGE ROW → img1.jpeg.png
   OCR completed
   Result: Tonal ORC ol
এখন ঘৰ _
Row 3: IMAGE ROW → img2.jpeg.png
   OCR completed
   Result: নাবেই এতিয়া একমাত্ৰ
ভৰসা যোৰাবাটত
Row 4: IMAGE ROW → img3.jpeg.png
   OCR completed
   Result: পুনৰ কৃত্ৰিম বানত ডুবিল
মহানগৰী,এয়া বশিষ্ঠৰ দৃশ্য
Row 5: IMAGE ROW → img4.jpeg.png
   OCR completed
   Result: बाढ़ पीड़ितों के लिए भोजन और दवा
की तत्काल आवश्यकता __
Row 6: TEXT ROW → OCR skipped
Row 7: TEXT ROW → OCR skipped
Row 8: IMAGE ROW → img5.jpeg.png
   OCR completed
   Result: SS OPI RC ললে
Row 9: TEXT ROW → OCR skipped
Row 10: TEXT ROW → OCR skipped
Row 11: TEXT ROW → OCR skipped
Row 12: TEXT ROW → OCR skipped
Row 13: TEXT ROW → OCR skipped
Row 14: TEXT ROW → OCR skipped
Row 15: IMAGE ROW → img6.jpeg.png
   OCR completed
 

,Id,Source,Text,Image,Extracted Text,Language,Code-Mixed,Translated Text
0,1,X,,img1.jpeg.png,Tonal ORC ol\nএখন ঘৰ _,NaN,NaN,NaN
1,2,X,,img2.jpeg.png,নাবেই এতিয়া একমাত্ৰ\nভৰসা যোৰাবাটত,NaN,NaN,NaN
2,3,X,,img3.jpeg.png,"পুনৰ কৃত্ৰিম বানত ডুবিল\nমহানগৰী,এয়া বশিষ্ঠৰ ...",NaN,NaN,NaN
3,4,X,,img4.jpeg.png,बाढ़ पीड़ितों के लिए भोजन और दवा\nकी तत्काल आव...,NaN,NaN,NaN
4,5,X,"তেওঁলোকৰোঁ আছিল এখন ঘৰ, আছিল বহু সপোন। এতিয়া ...",,,NaN,NaN,NaN
5,6,X,Guwahati r Anil Nagar fully underwater. Water ...,,,NaN,NaN,NaN
6,7,X,,img5.jpeg.png,SS OPI RC ললে,NaN,NaN,NaN
7,8,X,মৰিগাঁও জিলাৰ ভূৰাগাঁৱত ব্ৰহ্মপুত্ৰৰ মথাউৰি ভা...,,,NaN,NaN,NaN
8,9,X,असम के धेमाजी में बाढ़ की स्थिति गंभीर। राष्ट्...,,,NaN,NaN,NaN
9,10,X,The water levels of the Jia Bharali and Puthim...,,,NaN,NaN,NaN
